In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import  pyspark.sql.functions as fun 
import numpy as np
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
sc = spark.sparkContext

Create an RDD from a list of numbers (1,50) using numpy methods

In [4]:
data = np.arange(1,50)
rdd2 = sc.parallelize(data)

Find the sum, average, maximum, minimum, and count

In [5]:
#Sum
rdd3 = rdd2.sum()
rdd3

1225

In [6]:
#avg
rdd3 = rdd2.mean()
rdd3

25.0

In [7]:
#Count
rdd3 = rdd2.count()
rdd3


49

In [8]:
#Min
rdd3 = rdd2.min()
rdd3

1

In [9]:
#Max
rdd3 = rdd2.max()
rdd3

49

Count how many numbers are even vs. odd.

In [31]:
evenNum = rdd2.filter(lambda x : x%2==0).count()
oddNum = rdd2.filter(lambda x : x%2!=0).count()
print(evenNum)
print(oddNum)

24
25


You have the following data of people info ('Name', 'Age'), answer the following questions

In [11]:
people_data = [("Nada ", 25), ("Mona", 30), ("Ahmed", 35), ("Khaled", 40),("Ahmed", 35), ('Nada ', 25)]
rdd_people = sc.parallelize(people_data)
rdd_people.collect()

[('Nada ', 25),
 ('Mona', 30),
 ('Ahmed', 35),
 ('Khaled', 40),
 ('Ahmed', 35),
 ('Nada ', 25)]

Find the oldest person

In [12]:
oldest_person = rdd_people.max(lambda x: x[1])
oldest_person

('Khaled', 40)

Compute the average age

In [13]:
average_age = rdd_people.map(lambda x: x[1]).mean()
average_age

31.666666666666668

Group all the names by their age

In [14]:
name_by_age = rdd_people.groupBy(lambda x: x[1]).mapValues(lambda vals: [v[0] for v in vals])
print(name_by_age.collect())

[(35, ['Ahmed', 'Ahmed']), (40, ['Khaled']), (25, ['Nada ', 'Nada ']), (30, ['Mona'])]


Take the following text and put it in a text file named russia.txt and load it into rdd

"Russia is the largest country in the world by land area
Moscow is the capital city of Russia
The Russian language is one of the most widely spoken languages in the world
Russia is known for its rich history and culture
The Trans-Siberian Railway is the longest railway line in the world
Russia has a strong tradition in literature, music and ballet
The country is famous for its cold winters and vast landscapes
Russia is a major player in global energy production
"

In [15]:
rdd_russia = sc.textFile("/data/russia.txt")
rdd_russia.collect()

['"Russia is the largest country in the world by land area',
 'Moscow is the capital city of Russia',
 'The Russian language is one of the most widely spoken languages in the world',
 'Russia is known for its rich history and culture',
 'The Trans-Siberian Railway is the longest railway line in the world',
 'Russia has a strong tradition in literature, music and ballet',
 'The country is famous for its cold winters and vast landscapes',
 'Russia is a major player in global energy production"']

Count the total number of lines.

In [16]:
rdd_russia.count()

8

Count how many lines contain the word "Russia"

In [17]:
rdd_russia_count = rdd_russia.filter(lambda x: "Russia" in x).count()
rdd_russia_count

6

Find the most 5 frequent word in the file.

In [18]:
words_rdd = rdd_russia.flatMap(lambda x: x.split())
word_counts = words_rdd.map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)
top_5_words = word_counts.sortBy(lambda item: item[1], ascending=False).take(5)
print(top_5_words)

[('is', 7), ('the', 7), ('in', 5), ('Russia', 4), ('world', 3)]


Tokenize words

In [33]:
words = rdd_russia.flatMap(lambda line: line.split())
words = words.map(lambda w: w.lower())
words.collect()

['"russia',
 'is',
 'the',
 'largest',
 'country',
 'in',
 'the',
 'world',
 'by',
 'land',
 'area',
 'moscow',
 'is',
 'the',
 'capital',
 'city',
 'of',
 'russia',
 'the',
 'russian',
 'language',
 'is',
 'one',
 'of',
 'the',
 'most',
 'widely',
 'spoken',
 'languages',
 'in',
 'the',
 'world',
 'russia',
 'is',
 'known',
 'for',
 'its',
 'rich',
 'history',
 'and',
 'culture',
 'the',
 'trans-siberian',
 'railway',
 'is',
 'the',
 'longest',
 'railway',
 'line',
 'in',
 'the',
 'world',
 'russia',
 'has',
 'a',
 'strong',
 'tradition',
 'in',
 'literature,',
 'music',
 'and',
 'ballet',
 'the',
 'country',
 'is',
 'famous',
 'for',
 'its',
 'cold',
 'winters',
 'and',
 'vast',
 'landscapes',
 'russia',
 'is',
 'a',
 'major',
 'player',
 'in',
 'global',
 'energy',
 'production"']

Remove stopwords (a, the, is, to, in, of). 

In [19]:
stopwords = {"a", "the", "is", "to", "in", "of"}
filtered_words = words_rdd.filter(lambda w: w not in stopwords)
filtered_words.collect()

['"Russia',
 'largest',
 'country',
 'world',
 'by',
 'land',
 'area',
 'Moscow',
 'capital',
 'city',
 'Russia',
 'The',
 'Russian',
 'language',
 'one',
 'most',
 'widely',
 'spoken',
 'languages',
 'world',
 'Russia',
 'known',
 'for',
 'its',
 'rich',
 'history',
 'and',
 'culture',
 'The',
 'Trans-Siberian',
 'Railway',
 'longest',
 'railway',
 'line',
 'world',
 'Russia',
 'has',
 'strong',
 'tradition',
 'literature,',
 'music',
 'and',
 'ballet',
 'The',
 'country',
 'famous',
 'for',
 'its',
 'cold',
 'winters',
 'and',
 'vast',
 'landscapes',
 'Russia',
 'major',
 'player',
 'global',
 'energy',
 'production"']

Count the frequency of each word

In [20]:
words_frequency = filtered_words.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)
words_frequency.collect()

[('"Russia', 1),
 ('largest', 1),
 ('country', 2),
 ('world', 3),
 ('by', 1),
 ('land', 1),
 ('area', 1),
 ('capital', 1),
 ('Russia', 4),
 ('language', 1),
 ('most', 1),
 ('widely', 1),
 ('known', 1),
 ('for', 2),
 ('history', 1),
 ('and', 3),
 ('Trans-Siberian', 1),
 ('Railway', 1),
 ('line', 1),
 ('literature,', 1),
 ('music', 1),
 ('famous', 1),
 ('cold', 1),
 ('winters', 1),
 ('landscapes', 1),
 ('player', 1),
 ('energy', 1),
 ('production"', 1),
 ('Moscow', 1),
 ('city', 1),
 ('The', 3),
 ('Russian', 1),
 ('one', 1),
 ('spoken', 1),
 ('languages', 1),
 ('its', 2),
 ('rich', 1),
 ('culture', 1),
 ('longest', 1),
 ('railway', 1),
 ('has', 1),
 ('strong', 1),
 ('tradition', 1),
 ('ballet', 1),
 ('vast', 1),
 ('major', 1),
 ('global', 1)]

In [21]:
schema = 'id integer, name string, age integer, salary integer' 
data = [
    (1, "Ali", 25, 4000),
    (2, "Mariam", 30, 6000),
    (3, "Omar", 35, 7000),
    (4, "Sara", 28, 5000),
    (5, "Omar", 25, 6500),
    (6, "Mariam", 26, 7500)
]

df = spark.createDataFrame(data,schema)

Show schema and first 2 rows

In [22]:
df.show(2)

+---+------+---+------+
| id|  name|age|salary|
+---+------+---+------+
|  1|   Ali| 25|  4000|
|  2|Mariam| 30|  6000|
+---+------+---+------+
only showing top 2 rows



Select only name and salary

In [23]:
selected_info = df.select("name", "salary")
selected_info.show()

+------+------+
|  name|salary|
+------+------+
|   Ali|  4000|
|Mariam|  6000|
|  Omar|  7000|
|  Sara|  5000|
|  Omar|  6500|
|Mariam|  7500|
+------+------+



Find the average salary

In [24]:
df.agg({'salary': 'avg'}).show()

+-----------+
|avg(salary)|
+-----------+
|     6000.0|
+-----------+



Filter employees older than 28

In [25]:
df.filter("age > 28").show()

+---+------+---+------+
| id|  name|age|salary|
+---+------+---+------+
|  2|Mariam| 30|  6000|
|  3|  Omar| 35|  7000|
+---+------+---+------+



Count distinct values in the name column

In [26]:
df.select("name").distinct().count()


4

Group by a the name column and find average salary

In [27]:
df.groupBy("name").agg(fun.avg("salary").alias("Average")).show()

+------+-------+
|  name|Average|
+------+-------+
|   Ali| 4000.0|
|Mariam| 6750.0|
|  Omar| 6750.0|
|  Sara| 5000.0|
+------+-------+



In [28]:
df1 = spark.read.csv("/data/NullData.csv", header=True, inferSchema=True) #this file in shared folder
df1.show()

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp2| NULL| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



Find the avg sales 

In [29]:
df1.select(fun.avg("Sales")).show()

+----------+
|avg(Sales)|
+----------+
|     400.5|
+----------+



Replace null name with 'Unknown' and sales with the avg sales of the column 